In [ ]:
!pip install langchain langchain-community langchain-huggingface faiss-gpu sentence-transformers ranx bitsandbytes accelerate

In [ ]:
import os
import torch
from typing import List, Dict, Any
from datasets import Dataset
from ranx import Qrels, Run, evaluate

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.llms import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

# ==========================================
# 1. 환경 및 하드웨어 가속 설정
# ==========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"현재 사용 중인 하드웨어 장치: {device}")

# ==========================================
# 2. 가상의 RFP 테스트 데이터셋 구성
# ==========================================
# 실제 서비스 시에는 전처리한 문서 조각들을 여기에 매핑하세요.
mock_rfp_chunks = [
    Document(page_content="제5조(보안 관리): 본 사업의 개인정보 저장 시 AES-256 알고리즘으로 양방향 암호화 조치해야 하며, 비밀번호는 SHA-256 단방향 해시 처리한다.", metadata={"chunk_id": "chunk_001"}),
    Document(page_content="제12조(품질 보증): 시스템 가동 후 무상 유지보수 기간은 12개월로 하며, 월간 가동률은 99.9% 이상을 상시 유지하여야 한다.", metadata={"chunk_id": "chunk_002"}),
    Document(page_content="제23조(데이터 백업): 운영 데이터베이스 백업은 매일 자정(00:00)에 증분 백업을 수행하며, 백업본은 최소 3개월간 보관한다.", metadata={"chunk_id": "chunk_003"})
]

# Retriever 검증을 위한 평가 골든셋 (질문 - 정답 청크 ID 매핑)
test_dataset = [
    {
        "query_id": "q_1",
        "query": "비밀번호 암호화 표준 알고리즘 규칙이 어떻게 되나요?",
        "ground_truth_ids": ["chunk_001"]
    },
    {
        "query_id": "q_2",
        "query": "시스템 가동률 조건과 무상 유지보수 기간을 알려주세요.",
        "ground_truth_ids": ["chunk_002"]
    }
]

# ==========================================
# 3. 임베딩 모델 로드 및 FAISS Vector DB 빌드
# ==========================================
print("\n[1/4] 한국어 특화 'nlpai-lab/KURE-v1' 임베딩 모델 로드 중...")
embedding_model_name = "nlpai-lab/KURE-v1"
embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs={'device': device},
    encode_kwargs={'normalize_embeddings': True}
)

print("[2/4] FAISS Vector DB 인덱스 생성 중...")
vectorstore = FAISS.from_documents(mock_rfp_chunks, embeddings)
# 1차 후보군을 5개 추출하는 기본 Retriever 선언
faiss_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# ==========================================
# 4. Qwen2-7B-Instruct 4비트 양자화 로드 (코랩 최적화)
# ==========================================
print("[3/4] Qwen2-7B-Instruct 모델 양자화 로드 중 (시간이 다소 소요될 수 있습니다)...")
llm_model_id = "Qwen/Qwen2-7B-Instruct"

# 코랩 무료 T4 VRAM(16GB) 환경에서 안정적으로 구동하기 위한 4비트 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(llm_model_id)
model = AutoModelForCausalLM.from_pretrained(
    llm_model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1
)
llm = HuggingFacePipeline(pipeline=pipe)

# ==========================================
# 5. Ranx 기반 정량 평가 모듈 정의 (함수화)
# ==========================================
def evaluate_retriever_performance(
    test_data: List[Dict[str, Any]],
    retriever: Any,
    metrics: List[str] = ["recall@1", "recall@3", "recall@5", "mrr", "ndcg@5"]
) -> Dict[str, float]:
    """
    FAISS Retriever 파이프라인의 검색 순위 퀄리티를 Ranx로 정량 평가합니다.
    """
    qrels_dict = {}
    run_dict = {}
    
    print("\n🚀 Ranx 평가를 위한 Retriever 추론 및 스코어 매핑 시작...")
    
    for idx, item in enumerate(test_data):
        q_id = item.get("query_id")
        query_text = item.get("query")
        gt_ids = item.get("ground_truth_ids")
        
        # 실제 정답 매핑
        qrels_dict[q_id] = {str(gt_id): 1 for gt_id in gt_ids}
        
        # FAISS 유사도 검색 수행
        retrieved_docs = retriever.invoke(query_text)
        
        # Ranx 데이터 포맷에 맞춰 문서 점수 산출
        query_run_results = {}
        for rank, doc in enumerate(retrieved_docs):
            chunk_id = str(doc.metadata.get("chunk_id", f"unknown_{rank}"))
            # FAISS 유사도 스코어가 명시되지 않은 경우 순위 기반 패널티 스코어 부여
            score = doc.metadata.get("relevance_score", 1.0 / (rank + 1))
            query_run_results[chunk_id] = score
            
        run_dict[q_id] = query_run_results

    # Ranx 연산 실행
    qrels = Qrels(qrels_dict)
    run = Run(run_dict)
    results = evaluate(qrels, run, metrics)
    
    return dict(results) if isinstance(results, dict) else {metrics[0]: results}

# ==========================================
# 6. 전체 시스템 실행 및 최종 검증
# ==========================================
# 1) Ranx를 활용한 Retriever 검색 성능 지표 측정
eval_scores = evaluate_retriever_performance(
    test_data=test_dataset,
    retriever=faiss_retriever,
    metrics=["recall@1", "recall@3", "mrr"]
)

print("\n=============================================")
print("📊 [최종 결과] FAISS + KURE-v1 검색 성능 지표")
print("=============================================")
for metric, score in eval_scores.items():
    print(f"📌 {metric.upper().ljust(10)} : {score:.4f}")

# 2) 실제 Qwen2 LLM 연동 테스트 (샘플 1번 쿼리 적용)
print("\n=============================================")
print("🤖 Qwen2-7B-Instruct 최종 답변 생성 테스트")
print("=============================================")

sample_query = test_dataset[0]["query"]
search_docs = faiss_retriever.invoke(sample_query)
context_str = "\n".join([d.page_content for d in search_docs])

prompt = ChatPromptTemplate.from_template("""
<|im_start|>system
당신은 사내 RFP 전문 분석가입니다. 주어진 참고 문서를 바탕으로 질문에 사실대로 요약 답변하세요.
<|im_end|>
<|im_start|>user
[참고 문서]
{context}

[질문]
{question}
<|im_end|>
<|im_start|>assistant
""")

chain = prompt | llm | StrOutputParser()
ai_response = chain.invoke({"context": context_str, "question": sample_query})

print(f"질문: {sample_query}")
print(f"답변:\n{ai_response}")
